In [11]:
import numpy as np
import rasterio as rio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from affine import Affine
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FEATURE_DIR = RESULTS / "features"

In [3]:
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

In [7]:
treecover_2020_1_ = rio.open(DATA / 'tree_cover_2020_1.tif')
treecover_2020_2_ = rio.open(DATA / 'tree_cover_2020_2.tif')

In [8]:
treecover_2020_1 = treecover_2020_1_.read(1)
treecover_2020_2 = treecover_2020_2_.read(1)

In [9]:
treecover_2020_ = np.hstack([treecover_2020_1, treecover_2020_2])

In [10]:
treecover_2020 = np.nan_to_num(treecover_2020_).astype(np.int8)

In [14]:
forest_2020_1km = np.empty(country_raster_1km_.shape, dtype=np.uint8)

In [15]:
src_transfrom = Affine(1/120, 0.0, -180, 0.0, -1/120, 90)

In [16]:
reproject(
    source=treecover_2020,
    destination=forest_2020_1km,
    src_transform=src_transfrom,
    src_crs="EPSG:4326",
    dst_transform=country_raster_1km_.transform,
    dst_crs=country_raster_1km_.crs,
    resampling=Resampling.nearest,
)

(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], shape=(16922, 33844), dtype=uint8),
 Affine(1000.0, 0.0, -16922000.0,
        0.0, -1000.0, 8461000.0))

In [17]:
with rio.open(
    FEATURE_DIR / 'forest-2020-1km.tif',
    "w",
    driver="GTiff",
    height=country_raster_1km_.shape[0],
    width=country_raster_1km_.shape[1],
    count=1,
    dtype=forest_2020_1km.dtype,
    crs=country_raster_1km_.crs,
    transform=country_raster_1km_.transform,
    compress='lzw'
) as dst:
    dst.write(forest_2020_1km, 1)